# NaniGPT — Day 9-10: LoRA fine-tune Gemma 4 E4B for pill detection

**Goal:** lift per-slot pill detection accuracy from ~78% (zero-shot) to >95% via Unsloth QLoRA on 200 synthetic pill organizer photos. Unlocks the **$10K Unsloth special prize** and gives the writeup a concrete before-vs-after technical claim.

**Runtime:** Colab T4 (free) is sufficient. Training takes ~30-60 min depending on `max_steps`.

**Data:** 200 training + 30 eval pill organizer images, each with structured ground-truth labels. Generated synthetically with varied lighting, fill patterns, blur, rotation, and titles. Bundled in `training_set.zip` in the workspace.

## Step 1 — Install dependencies

In [ ]:
%%capture
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
    unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm \
    trl peft accelerate datasets

## Step 2 — Upload training_set.zip and unzip

In [ ]:
from google.colab import files
print('Upload training_set.zip from your workspace folder:')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
!unzip -qo {zip_name} -d /content/data/
# The zip preserves absolute paths — flatten
import os, shutil
for sub in ['training', 'eval']:
    src = '/content/data'
    found = []
    for root, _, files_ in os.walk(src):
        if root.endswith('/' + sub):
            found.append(root)
    if found and not os.path.exists(f'/content/{sub}'):
        shutil.move(found[0], f'/content/{sub}')
!ls /content/training | head -3
!ls /content/eval | head -3
print('Train images:', len([f for f in os.listdir('/content/training') if f.endswith('.jpg')]))
print('Eval images: ', len([f for f in os.listdir('/content/eval')     if f.endswith('.jpg')]))

## Step 3 — Load Gemma 4 E4B with Unsloth + apply vision LoRA adapters

In [ ]:
import torch
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name = 'unsloth/gemma-4-e4b-it',
    max_seq_length = 4096,
    load_in_4bit = True,
    full_finetuning = False,
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r              = 16,
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = 'none',
    random_state   = 3407,
)
model.print_trainable_parameters()

## Step 4 — Build the training dataset (image + prompt → JSON answer)

In [ ]:
import json, os
from PIL import Image
from datasets import Dataset

TRAIN_INDEX = '/content/training/index.jsonl'
TRAIN_DIR   = '/content/training'

rows = []
with open(TRAIN_INDEX) as f:
    for line in f:
        rows.append(json.loads(line))
print('Loaded', len(rows), 'training examples')

def to_messages(row):
    img = Image.open(os.path.join(TRAIN_DIR, row['image'])).convert('RGB')
    img.thumbnail((768, 768))
    return {
        'messages': [
            {'role': 'user', 'content': [
                {'type': 'image', 'image': img},
                {'type': 'text',  'text': row['prompt']},
            ]},
            {'role': 'assistant', 'content': [
                {'type': 'text',  'text': row['answer']},
            ]},
        ]
    }

train_ds = Dataset.from_list([to_messages(r) for r in rows])
print(train_ds)

## Step 5 — Configure SFTTrainer and train

T4 with 4-bit quantization + LoRA: per-step ~6-8 sec for image+text. `max_steps=120` is ~15-20 min. Bump to 200-300 for higher quality if you have time.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_ds,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps    = 120,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim         = 'adamw_8bit',
        weight_decay  = 0.01,
        lr_scheduler_type = 'linear',
        seed = 3407,
        output_dir = '/content/lora_out',
        remove_unused_columns = False,
        dataset_text_field = '',
        dataset_kwargs = {'skip_prepare_dataset': True},
        max_length = 2048,
        report_to = 'none',
    ),
)
trainer_stats = trainer.train()

## Step 6 — Save the LoRA adapter (this is what we ship with the submission)

In [ ]:
ADAPTER_DIR = '/content/nanigpt_pill_lora'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
!ls -lh {ADAPTER_DIR}
# Bundle for download
!cd /content && zip -qr nanigpt_pill_lora.zip nanigpt_pill_lora
!ls -lh /content/nanigpt_pill_lora.zip
from google.colab import files as _f
_f.download('/content/nanigpt_pill_lora.zip')

## Step 7 — Evaluation: base vs fine-tuned per-slot accuracy

**This is the writeup-defining section.** Run inference on the held-out 30 eval images, score per-slot, compare against the same images run through the **base** model (which we'd need to load separately). For simplicity we score only the fine-tuned model here and use the prior measurement (78%) as the baseline anchor.

In [ ]:
import re, json
from PIL import Image
from transformers import TextStreamer

def classify(img):
    msgs = [
        {'role': 'user', 'content': [
            {'type': 'image', 'image': img},
            {'type': 'text',  'text': 'List EVERY compartment in this weekly pill organizer (MON-SUN, AM and PM = 14 total). For each, output JSON: [{"day":"MON","ampm":"AM","has_pills":true}, ...]. Output ONLY the JSON array, nothing else.'},
        ]},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors='pt',
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=600, temperature=0.2, top_p=0.9, do_sample=False)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    m = re.search(r'\[.*\]', text, re.DOTALL)
    return json.loads(m.group(0)) if m else None

EVAL_DIR = '/content/eval'
EVAL_INDEX = '/content/eval/index.jsonl'

rows = [json.loads(l) for l in open(EVAL_INDEX)]
total = correct = 0
per_image = []

for row in rows:
    img = Image.open(os.path.join(EVAL_DIR, row['image'])).convert('RGB')
    img.thumbnail((768, 768))
    pred = classify(img) or []
    truth = json.loads(row['answer'])
    truth_map = {(s['day'], s['ampm']): s['has_pills'] for s in truth}
    pred_map  = {(s['day'], s['ampm']): s['has_pills'] for s in pred}
    img_correct = sum(1 for k in truth_map if pred_map.get(k) == truth_map[k])
    img_total   = len(truth_map)
    correct += img_correct
    total   += img_total
    per_image.append((row['image'], img_correct, img_total))

print(f'\n=== FINE-TUNED MODEL — held-out eval ===')
print(f'Per-slot accuracy: {correct}/{total} = {100*correct/total:.1f}%')
print(f'Per-image breakdown:')
for f, c, t in per_image[:10]:
    print(f'  {f}: {c}/{t}')
print('  ... (showing first 10 of 30)')

## What you should see

**Good outcome:** per-slot accuracy 92-98%. That's our writeup story:
> *"Zero-shot Gemma 4 E4B classified pill organizer compartments at 78-93% per-slot accuracy. After 120 steps of QLoRA fine-tuning on 200 synthetic photos with varied lighting and fill patterns (Unsloth, Colab T4, ~20 min), per-slot accuracy on a held-out 30-image set rose to **{XX}%**. The LoRA adapter (~80MB) is bundled with the submission and can be merged into the base Gemma 4 E4B for production deployment via MediaPipe LLM Inference on Android."*

**Acceptable outcome:** 85-92%. Still a publishable result, still buys us Unsloth eligibility.

**Bad outcome (<85%):** something is misconfigured. Likely culprits:
1. Image resize mismatch between train and eval — verify both paths use `thumbnail((768, 768))`
2. Vision layers not actually being adapted — verify `print_trainable_parameters()` shows millions, not zeros
3. Insufficient training steps — bump `max_steps` to 250

Save the printed `Per-slot accuracy` line — that's the number that goes in the writeup.